In [1]:
import os, sys
sys.path.insert(0, "/home/halzyon/Data/coding_work/machine_learning/accelerator/cf-rating-predictor")
os.chdir("/home/halzyon/Data/coding_work/machine_learning/accelerator/cf-rating-predictor")

In [2]:
from src.api.collector import CodeforcesAPICollector            
c = CodeforcesAPICollector()
c.fetch_problems();
c.fetch_contests();

2026-04-26 22:09:53,166 INFO src.api.collector: problem_api.json already exists (pass force=True to re-fetch)
2026-04-26 22:09:53,184 INFO src.api.collector: contests_api.json already exists (pass force=True to re-fetch)


In [3]:
from src.data.schema import build_merged_dataframe
df = build_merged_dataframe()                                   
df.shape      

2026-04-26 22:09:53,433 INFO src.data.schema: Merged DataFrame: 11155 rows, 13 columns


(11155, 13)

In [4]:
df.dtypes

problem_key                  str
contest_id                 Int64
problem_index                str
name                         str
problem_type                 str
points                   float64
rating                   float64
tags                      object
solved_count               Int64
contest_name                 str
contest_type                 str
contest_duration_secs      Int64
contest_start_time         Int64
dtype: object

In [5]:
df.head()

,problem_key,contest_id,problem_index,name,problem_type,points,rating,tags,solved_count,contest_name,contest_type,contest_duration_secs,contest_start_time
0,2225_G,2225,G,Simple Problem,PROGRAMMING,NaN,NaN,"[graphs, number theory]",53,Educational Codeforces Round 189 (Rated for Di...,ICPC,7200,1776782100
1,2225_F,2225,F,String Cutting,PROGRAMMING,NaN,NaN,"[binary search, brute force, greedy, hashing, ...",306,Educational Codeforces Round 189 (Rated for Di...,ICPC,7200,1776782100
2,2225_E,2225,E,Covering Points with Circles,PROGRAMMING,NaN,NaN,"[constructive algorithms, data structures, geo...",692,Educational Codeforces Round 189 (Rated for Di...,ICPC,7200,1776782100
3,2225_D,2225,D,Exceptional Segments,PROGRAMMING,NaN,NaN,"[bitmasks, brute force, math]",4687,Educational Codeforces Round 189 (Rated for Di...,ICPC,7200,1776782100
4,2225_C,2225,C,Red-Black Pairs,PROGRAMMING,NaN,NaN,"[dp, greedy]",8700,Educational Codeforces Round 189 (Rated for Di...,ICPC,7200,1776782100


In [6]:
from src.data.cleaner import clean_and_validation
labeled, unlabeled = clean_and_validation(df)
print(labeled.shape, unlabeled.shape)


2026-04-26 22:09:53,460 INFO src.data.cleaner: Start cleaning. No of rows: 11155
2026-04-26 22:09:53,462 INFO src.data.cleaner: After filter: 11155 rows
2026-04-26 22:09:53,464 INFO src.data.cleaner: Dropped duplicates (by problem_key): 11155 rows
2026-04-26 22:09:53,474 INFO src.data.cleaner: Labeled (rating known): 10864 rows
2026-04-26 22:09:53,475 INFO src.data.cleaner: Unlabeled (rating unknown): 291 rows
(10864, 15) (291, 15)


In [7]:
from pathlib import Path

Path("data/intermediate").mkdir(parents=True,exist_ok=True)
labeled.to_parquet("data/intermediate/labeled.parquet",index=False)
print(labeled.shape, unlabeled.shape)

(10864, 15) (291, 15)


In [8]:
from src.features.pipeline import build_feature_pipeline
build_feature_pipeline()

2026-04-26 22:09:53,542 INFO src.features.pipeline: Loaded labeled data: 10864 rows
2026-04-26 22:09:53,547 INFO src.features.pipeline: Split — Train: 6961 problems (1351 contests) | Val: 1844 (289) | Test: 2059 (291)
2026-04-26 22:09:53,552 INFO src.features.pipeline: Building features for variant A …
2026-04-26 22:09:53,632 INFO src.features.pipeline: Variant A: 16 features | train=6961 val=1844 test=2059
2026-04-26 22:09:53,633 INFO src.features.pipeline: Building features for variant B …
2026-04-26 22:09:54,357 INFO src.features.pipeline: Variant B: 55 features | train=6961 val=1844 test=2059
2026-04-26 22:09:54,357 INFO src.features.pipeline: Building features for variant C …
2026-04-26 22:09:55,079 INFO src.features.pipeline: Variant C: 57 features | train=6961 val=1844 test=2059
2026-04-26 22:09:55,079 INFO src.features.pipeline: Feature pipeline complete.


In [9]:
from src.models.trainer import train_all_models
results_df = train_all_models()
results_df

2026-04-26 22:09:55,725 INFO src.models.trainer: === Variant A ===
2026-04-26 22:09:55,731 INFO src.models.trainer:   Training mean_A …
2026-04-26 22:09:55,732 INFO src.models.trainer:     val MAE = 702.8
2026-04-26 22:09:55,733 INFO src.models.trainer:   Training median_A …
2026-04-26 22:09:55,733 INFO src.models.trainer:     val MAE = 696.0
2026-04-26 22:09:55,734 INFO src.models.trainer:   Training ridge_A …
2026-04-26 22:09:55,741 INFO src.models.trainer:     val MAE = 344.8
2026-04-26 22:09:55,742 INFO src.models.trainer:   Training lgbm_A …
2026-04-26 22:09:57,747 INFO src.models.trainer:     val MAE = 238.0
2026-04-26 22:09:57,747 INFO src.models.trainer:   Training xgb_A …
2026-04-26 22:09:58,844 INFO src.models.trainer:     val MAE = 238.6
2026-04-26 22:09:58,845 INFO src.models.trainer: === Variant B ===
2026-04-26 22:09:58,859 INFO src.models.trainer:   Training mean_B …
2026-04-26 22:09:58,860 INFO src.models.trainer:     val MAE = 702.8
2026-04-26 22:09:58,860 INFO src.mod

,model,variant,train_date,num_train_samples,num_val_samples,num_features,val_mae
13,lgbm,C,2026-04-27T02:10:03.783262+00:00,6961,1844,57,126.79
14,xgb,C,2026-04-27T02:10:04.990800+00:00,6961,1844,57,129.52
12,ridge,C,2026-04-27T02:10:01.808136+00:00,6961,1844,57,204.17
9,xgb,B,2026-04-27T02:10:01.783747+00:00,6961,1844,55,217.39
8,lgbm,B,2026-04-27T02:10:00.903076+00:00,6961,1844,55,234.70
3,lgbm,A,2026-04-27T02:09:57.746973+00:00,6961,1844,16,237.95
4,xgb,A,2026-04-27T02:09:58.844016+00:00,6961,1844,16,238.64
7,ridge,B,2026-04-27T02:09:58.870649+00:00,6961,1844,55,330.19
2,ridge,A,2026-04-27T02:09:55.741834+00:00,6961,1844,16,344.85
11,median,C,2026-04-27T02:10:01.800250+00:00,6961,1844,57,696.04


In [10]:
from src.evaluation.metrics import evaluate_all_models    
evaluate_all_models()

2026-04-26 22:10:05,227 INFO src.evaluation.metrics: Saved model_comparison.md
2026-04-26 22:10:05,313 INFO src.evaluation.metrics: Evaluation complete. Reports saved to reports


,MAE,RMSE,R2,MedAE,within_100,within_200,model,variant
0,689.28,811.68,-0.0001,680.05,0.0884,0.1632,mean,A
1,687.37,815.01,-0.0083,700.00,0.1282,0.1967,median,A
2,332.45,431.94,0.7168,249.31,0.1598,0.3351,ridge,A
3,238.28,340.89,0.8236,171.85,0.3409,0.5595,lgbm,A
4,240.41,365.38,0.7973,160.73,0.3643,0.5566,xgb,A
5,689.28,811.68,-0.0001,680.05,0.0884,0.1632,mean,B
6,687.37,815.01,-0.0083,700.00,0.1282,0.1967,median,B
7,322.80,419.56,0.7328,268.26,0.1948,0.3847,ridge,B
8,241.01,332.70,0.8320,179.01,0.3181,0.5454,lgbm,B
9,228.16,322.65,0.8420,167.00,0.3380,0.5614,xgb,B


In [ ]:
import pandas as pd

# Show top 20 features for lgbm variant C (best model)
fi = pd.read_csv("reports/feature_importance_lgbm_C.csv")
fi.head(20)